test call 

In [ ]:
import requests

In [ ]:
# Facility setup
facility = {
    "name": "TEST_LOCATION",
    "lat": 35.015,
    "lon": -90.015
}

In [ ]:
lat = facility["lat"]
lon = facility["lon"]

In [ ]:
facility_name = facility["name"].replace(" ", "_")

In [ ]:
# Determine local UTM CRS
def get_utm_epsg(lat, lon):
    """
    Return the appropriate UTM EPSG code
    for a latitude/longitude coordinate.
    """

    zone = int((lon + 180) / 6) + 1

    if lat >= 0:
        return 32600 + zone
    else:
        return 32700 + zone

In [ ]:
utm_epsg = get_utm_epsg(lat, lon)

In [ ]:
print("\nFACILITY")
print("Name:", facility["name"])
print("Latitude:", lat)
print("Longitude:", lon)
print("UTM CRS:", utm_epsg)

In [ ]:
# EPQS point query
url = "https://epqs.nationalmap.gov/v1/json"

In [ ]:
point_params = {
    "x": lon,
    "y": lat,
    "wkid": 4326,
    "units": "Meters",
    "includeDate": True
}

In [ ]:
response = requests.get(url, params=point_params)
response.raise_for_status()

In [ ]:
data = response.json()

In [ ]:
print("\nEPQS RESPONSE")
print(data)

In [ ]:
#searches the national map for DEM products
tnm_url = "https://tnmaccess.nationalmap.gov/api/v1/products"

In [ ]:
delta = 0.01

In [ ]:
bbox = [
    lon - delta,
    lat - delta,
    lon + delta,
    lat + delta
]

In [ ]:
params = {
    "bbox": ",".join(map(str, bbox)),
    "datasets": "Digital Elevation Model (DEM) 1 meter",
    "outputFormat": "JSON",
    "max": 20
}

In [ ]:
response = requests.get(tnm_url, params=params)
response.raise_for_status()

In [ ]:
products = response.json()

In [ ]:
print("Bounding box:", bbox)
print("Total products:", products.get("total"))

In [ ]:
#inspects finidning 
items = products.get("items", [])

In [ ]:
print("Returned:", len(items))

In [ ]:
for item in items[:5]:
    print("\n--- PRODUCT ---")
    print("Title:", item.get("title"))
    print("Date:", item.get("dateCreated"))
    print("Format:", item.get("format"))
    print("Size:", item.get("sizeInBytes"))
    print("Download:", item.get("downloadURL"))

In [ ]:
#inspects the 7 files 
for i, item in enumerate(items):
    print(f"\n--- PRODUCT {i} ---")
    print("Title:       ", item.get("title"))
    print("Date created:", item.get("dateCreated"))
    print("Format:      ", item.get("format"))
    print("Size (MB):   ", round(item.get("sizeInBytes", 0) / 1e6, 2))
    print("Bounding box:", item.get("boundingBox"))

In [ ]:
#keys for one result
print("\nAvailable metadata fields:")
print(items[0].keys())

inspects metadata for candidate product

In [ ]:
fields = [
    "title",
    "sourceName",
    "sourceOriginName",
    "publicationDate",
    "lastUpdated",
    "dateCreated",
    "extent",
    "datasets",
    "bestFitIndex",
    "metaUrl",
    "vendorMetaUrl",
]

In [ ]:
for field in fields:
    print(f"\n{field}:")
    print(candidate.get(field))

In [ ]:
#spatial check
def contains_point(item, lon, lat):
    bounds = item["boundingBox"]

    return (
        bounds["minX"] <= lon <= bounds["maxX"]
        and bounds["minY"] <= lat <= bounds["maxY"]
    )

In [ ]:
covering_products = [
    item for item in items
    if contains_point(item, lon, lat)
]

In [ ]:
print("\nProducts containing facility point:", len(covering_products))

In [ ]:
for item in covering_products:
    print(
        item["title"],
        "|",
        round(item["sizeInBytes"] / 1e6, 2),
        "MB"
    )

records point level 3DEP metadata

In [ ]:
point_metadata = {
    "longitude": lon,
    "latitude": lat,
    "elevation_m": float(data["value"]),
    "resolution_m": data["resolution"],
    "acquisition_date": data["attributes"].get("AcquisitionDate"),
    "num_1m_products_found": products.get("total"),
    "num_products_covering_point": len(covering_products),
}

In [ ]:
print("\n3DEP POINT SUMMARY")
for key, value in point_metadata.items():
    print(f"{key}: {value}")

eequest + download 3DEP raster directly

In [ ]:
from pathlib import Path

In [ ]:
export_url = (
    "https://elevation.nationalmap.gov/"
    "arcgis/rest/services/3DEPElevation/ImageServer/exportImage"
)

In [ ]:
raster_params = {
    "bbox": ",".join(map(str, bbox)),
    "bboxSR": 4326,
    "imageSR": 4326,
    "size": "1000,1000",
    "format": "tiff",
    "pixelType": "F32",
    "interpolation": "RSP_BilinearInterpolation",
    "f": "image"          # <-- important change
}

In [ ]:
output_dir = Path("data/raw/usgs_3dep")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
facility_name = facility["name"].replace(" ", "_")
dem_path = output_dir / f"{facility_name}_dem.tif"
response = requests.get(
    export_url,
    params=raster_params,
    timeout=120
)

In [ ]:
print("\nRASTER REQUEST")
print("Status:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Size (MB):", round(len(response.content) / 1e6, 2))

In [ ]:
response.raise_for_status()

In [ ]:
with open(dem_path, "wb") as f:
    f.write(response.content)

In [ ]:
print("Saved:", dem_path)

Inspects GeoTIFF

In [ ]:
import rasterio

In [ ]:
with rasterio.open(dem_path) as src:
    print("\nDEM METADATA")
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bands:", src.count)
    print("Dtype:", src.dtypes)
    print("NoData:", src.nodata)
    print("Bounds:", src.bounds)
    print("Transform:", src.transform)
    print("Resolution:", src.res)

    elevation = src.read(1)

In [ ]:
#numerical sanity check
import numpy as np

In [ ]:
valid = elevation[np.isfinite(elevation)]

In [ ]:
print("\nELEVATION SUMMARY")
print("Valid pixels:", valid.size)
print("Min:", valid.min())
print("Max:", valid.max())
print("Mean:", valid.mean())
print("Std:", valid.std())

In [ ]:
print("\nPercentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"{p}%:", np.percentile(valid, p))

reproject DEM to meter based CRS

In [ ]:
from rasterio.warp import calculate_default_transform, reproject, Resampling

In [ ]:
projected_path = output_dir / f"{facility_name}_dem_utm.tif"

In [ ]:
dst_crs = f"EPSG:{get_utm_epsg(lat, lon)}"
nodata_value = -9999.0

In [ ]:
with rasterio.open(dem_path) as src:

    transform, width, height = calculate_default_transform(
        src.crs,
        dst_crs,
        src.width,
        src.height,
        *src.bounds
    )

    # Copy raster metadata
    raster_profile = src.profile.copy()

    raster_profile.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height,
        "nodata": nodata_value
    })

    with rasterio.open(projected_path, "w", **raster_profile) as dst:

        reproject(
            source=rasterio.band(src, 1),
            destination=rasterio.band(dst, 1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            src_nodata=src.nodata,
            dst_nodata=nodata_value,
            resampling=Resampling.bilinear
        )

In [ ]:
print("\nSaved projected DEM:")
print(projected_path)

In [ ]:
#inspects the reproject
with rasterio.open(projected_path) as src:
    print("\nPROJECTED DEM METADATA")
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)
    print("Transform:", src.transform)

Calculate slope

In [ ]:
with rasterio.open(projected_path) as src:
    elevation_utm = src.read(1).astype("float32")
    x_res, y_res = src.res

    elevation_utm[elevation_utm == src.nodata] = np.nan

In [ ]:
valid_mask = np.isfinite(elevation_utm)

In [ ]:
filled = np.where(
    valid_mask,
    elevation_utm,
    np.nanmedian(elevation_utm)
)

In [ ]:
dz_dy, dz_dx = np.gradient(
    filled,
    y_res,
    x_res
)

In [ ]:
slope_rad = np.arctan(
    np.sqrt(dz_dx**2 + dz_dy**2)
)

In [ ]:
slope_deg = np.degrees(slope_rad)

In [ ]:
slope_deg[~valid_mask] = np.nan

In [ ]:
#stats from valid pixels 
valid_slope = slope_deg[np.isfinite(slope_deg)]

In [ ]:
print("\nSLOPE SUMMARY")
print("Min:", valid_slope.min())
print("Max:", valid_slope.max())
print("Mean:", valid_slope.mean())
print("Std:", valid_slope.std())

In [ ]:
print("\nPercentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"{p}%:", np.percentile(valid_slope, p))

In [ ]:
#Visualize the slope
import matplotlib.pyplot as plt

In [ ]:
# Elevation
plt.figure(figsize=(8, 7))

In [ ]:
plt.imshow(elevation_utm)
plt.colorbar(label="Elevation (m)")

In [ ]:
plt.title("USGS 3DEP Elevation")
plt.xlabel("Pixel X")
plt.ylabel("Pixel Y")

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
# Slope
plt.figure(figsize=(8, 7))

In [ ]:
plt.imshow(slope_deg)
plt.colorbar(label="Slope (degrees)")

In [ ]:
plt.title("USGS 3DEP Slope")
plt.xlabel("Pixel X")
plt.ylabel("Pixel Y")

In [ ]:
plt.tight_layout()
plt.show()